# Plan D per-deck value NN 学習 (= 2026-05-18 夜)

**Plan F 失敗 / Plan D single でも -2.5pt** だが、 **deck 毎 ±60pt 振れ幅** で argmax boundary 動くこと検証済 (= 重要 validation)。

「Plan D single は 全 deck 混在で 学習 → deck-specific 戦術と 競合してゼロサム」 が 原因仮説。
**deck 専用 NN × 16** で 解決を 期待。

入力: `MyDrive/onepiece_nn/mcts_rollout_by_deck.tar.gz` (= ~900 KB、 16 個 jsonl 入り、 自動展開)
出力: `MyDrive/onepiece_nn/value_nn_per_deck/<slug>.pt` × 16

学習時間目安: T4 GPU で 全 16 NN ~5-10 分

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available()

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, tarfile
import numpy as np

WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAP_DIR = os.path.join(WORK_DIR, 'mcts_rollout_by_deck')
OUT_DIR = os.path.join(WORK_DIR, 'value_nn_per_deck')
os.makedirs(OUT_DIR, exist_ok=True)

# tar.gz が Drive に あれば 自動展開 (= ohtsuki さん が 個別 upload 不要)
tar_path = os.path.join(WORK_DIR, 'mcts_rollout_by_deck.tar.gz')
if not os.path.isdir(SNAP_DIR) and os.path.exists(tar_path):
    print(f'extracting {tar_path}...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(WORK_DIR)
    print('extracted to', SNAP_DIR)

assert os.path.isdir(SNAP_DIR), f'snap dir 不在: {SNAP_DIR}'
snap_files = sorted([f for f in os.listdir(SNAP_DIR) if f.endswith('.jsonl')])
print(f'found {len(snap_files)} deck snap files:')
for f in snap_files:
    path = os.path.join(SNAP_DIR, f)
    sz = os.path.getsize(path) // 1024
    print(f'  {f} ({sz} KB)')

In [ ]:
import torch.nn as nn
device = torch.device('cuda')

STATE_DIM = 172

class ValueNN(nn.Module):
    def __init__(self, input_dim=172, hidden=256, dropout=0.15):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden // 2, 1)
    def forward(self, x):
        return torch.sigmoid(self.head(self.body(x))).squeeze(-1)

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 50
BATCH = 128
LR = 1e-3

summary = {}
t_global = time.time()

for fname in snap_files:
    slug = fname.replace('.jsonl', '')
    path = os.path.join(SNAP_DIR, fname)
    snaps = []
    with open(path) as f:
        for line in f:
            try:
                d = json.loads(line)
                if 'state_encoded' in d and 'p_win' in d:
                    snaps.append(d)
            except: pass
    if len(snaps) < 50:
        print(f'[{slug}] {len(snaps)} snaps、 不足で SKIP')
        continue

    X = np.zeros((len(snaps), STATE_DIM), dtype=np.float32)
    Y = np.zeros((len(snaps),), dtype=np.float32)
    for i, s in enumerate(snaps):
        se = s['state_encoded']
        if len(se) >= STATE_DIM:
            X[i, :] = se[:STATE_DIM]
        else:
            X[i, :len(se)] = se
        Y[i] = float(s['p_win'])
    Xt = torch.from_numpy(X).to(device)
    Yt = torch.from_numpy(Y).to(device)

    model = ValueNN().to(device)
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    criterion = nn.MSELoss()

    train_ds = TensorDataset(Xt, Yt)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

    t_deck = time.time()
    final_loss = None
    for epoch in range(EPOCHS):
        ep_loss = 0.0
        nb = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            ep_loss += loss.item()
            nb += 1
        scheduler.step()
        final_loss = ep_loss / nb if nb else None

    out_path = os.path.join(OUT_DIR, f'{slug}.pt')
    torch.save(model.state_dict(), out_path)
    summary[slug] = {'snaps': len(snaps), 'mean_p_win': float(np.mean(Y)), 'final_loss': final_loss, 'elapsed_s': time.time()-t_deck}
    print(f'[{slug}] {len(snaps)} snaps, mean_p_win={np.mean(Y):.3f}, final_loss={final_loss:.4f}, {time.time()-t_deck:.1f}s → {out_path}')

print(f'\n=== TOTAL {time.time()-t_global:.0f}s ===')
with open(os.path.join(OUT_DIR, '_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

## 完了

`MyDrive/onepiece_nn/value_nn_per_deck/` 配下 16 個 .pt を ローカル `db/value_nn_per_deck/` へ download してください。